# Basic Feature Engineering: Bike Sharing

This notebook uses simple rules that are easy to explain. It works from the local `data/hour.csv` file and does not use cyclic encodings, lag features, or other advanced transformations.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

data = pd.read_csv("../data/hour.csv")
data.head()

## 1. Create `time_period`

The hour ranges are deliberately simple:

- **Night:** 00:00-05:59 (`hr` 0-5)
- **Morning:** 06:00-11:59 (`hr` 6-11)
- **Afternoon:** 12:00-17:59 (`hr` 12-17)
- **Evening:** 18:00-23:59 (`hr` 18-23)

In [ ]:
def get_time_period(hour):
    if 6 <= hour <= 11:
        return "morning"
    if 12 <= hour <= 17:
        return "afternoon"
    if 18 <= hour <= 23:
        return "evening"
    return "night"


data["time_period"] = data["hr"].apply(get_time_period)

## 2. Create `day_type`

A holiday is checked first because a holiday can also fall on a weekday. The remaining rows are split into ordinary working days and weekends.

In [ ]:
def get_day_type(row):
    if row["holiday"] == 1:
        return "holiday"
    if row["workingday"] == 1:
        return "ordinary_working_day"
    return "weekend"


data["day_type"] = data.apply(get_day_type, axis=1)

## 3. Convert normalized temperature to Celsius

In the UCI dataset, `temp` is normalized by dividing Celsius temperature by 41. Therefore, multiplying by 41 reverses that normalization:

`temperature_celsius = temp * 41`

In [ ]:
data["temperature_celsius"] = data["temp"] * 41

## 4. Create `demand_level`

The thresholds are based on the dataset itself rather than arbitrary numbers. The 33rd percentile is the low/medium boundary, and the 67th percentile is the medium/high boundary. This creates three groups of approximately equal size:

- **Low:** `cnt` at or below the 33rd percentile
- **Medium:** above the 33rd percentile and at or below the 67th percentile
- **High:** above the 67th percentile

In [ ]:
low_demand_threshold = data["cnt"].quantile(1 / 3)
high_demand_threshold = data["cnt"].quantile(2 / 3)

data["demand_level"] = pd.cut(
    data["cnt"],
    bins=[-float("inf"), low_demand_threshold, high_demand_threshold, float("inf")],
    labels=["low", "medium", "high"],
    include_lowest=True,
)

In [ ]:
print(f"Low/medium threshold: {low_demand_threshold:.1f} rentals")
print(f"Medium/high threshold: {high_demand_threshold:.1f} rentals")
print(data[["hr", "time_period", "day_type", "temp", "temperature_celsius", "cnt", "demand_level"]].head())
print("\nDemand-level counts:")
print(data["demand_level"].value_counts().sort_index())

The engineered DataFrame is now available as `data` for exploration in this notebook.

## 5. Line Chart: Hourly Demand

This line chart shows the average number of rentals for each hour. Peaks identify the busiest hours, while dips identify low-demand hours.

In [ ]:
hourly_demand = data.groupby("hr", as_index=False)["cnt"].mean()

plt.figure(figsize=(9, 5))
sns.lineplot(data=hourly_demand, x="hr", y="cnt", marker="o")
plt.title("Average Bike Rentals by Hour")
plt.xlabel("Hour of day")
plt.ylabel("Average rentals")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

## 6. Heatmap: Day Type and Hour

This heatmap compares the average demand for each `day_type` and hour combination. Darker cells represent higher average rental counts.

In [ ]:
day_hour_demand = data.pivot_table(
    index="day_type", columns="hr", values="cnt", aggfunc="mean"
)

plt.figure(figsize=(12, 4))
sns.heatmap(day_hour_demand, cmap="YlGnBu", annot=False)
plt.title("Average Rentals by Day Type and Hour")
plt.xlabel("Hour of day")
plt.ylabel("Day type")
plt.tight_layout()
plt.show()

## 7. Box Plot: Rentals by Season

This box plot compares the distribution of rental counts across seasons. The median, spread, and unusually high or low values can be compared between groups.

In [ ]:
season_labels = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}
season_data = data.assign(season_name=data["season"].map(season_labels))

plt.figure(figsize=(8, 5))
sns.boxplot(data=season_data, x="season_name", y="cnt")
plt.title("Rental Count Distribution by Season")
plt.xlabel("Season")
plt.ylabel("Rental count")
plt.tight_layout()
plt.show()

## 8. Scatter Plot: Temperature and Rentals

This scatter plot examines whether rental counts tend to change as temperature in Celsius changes. Each point represents one hourly observation.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=data,
    x="temperature_celsius",
    y="cnt",
    alpha=0.35,
    s=25,
)
plt.title("Temperature and Bike Rental Counts")
plt.xlabel("Temperature (°C)")
plt.ylabel("Rental count")
plt.tight_layout()
plt.show()

## 9. Histogram: Overall Rental Distribution

This histogram shows the overall spread of hourly rental counts. It helps reveal whether most observations have low counts and whether the distribution has a long high-demand tail.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=data, x="cnt", bins=30, kde=False)
plt.title("Distribution of Hourly Rental Counts")
plt.xlabel("Rental count")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()

# 10. Correlogram: Multivariate Associations

In [ ]:
# Select only the quantitative numerical columns for correlation
numeric_cols = ["temperature_celsius", "hum", "windspeed", "casual", "registered", "cnt"]
correlation_matrix = data[numeric_cols].corr()

# Plot the Correlogram
plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix, 
    annot=True,          # Shows the actual correlation numbers
    cmap="coolwarm",     # Blue for negative, Red for positive
    fmt=".2f",           # Rounds to 2 decimal places
    linewidths=0.5
)
plt.title("Correlogram of Weather and Rental Variables")
plt.tight_layout()
plt.show()

## Final Insights and Conclusion

### Insights from the five charts

1. **Line chart:** Average demand changes substantially across the day. Rental activity is generally highest during the morning and late afternoon commuting periods, while the early morning hours have lower demand.

2. **Heatmap:** The combination of day type and hour affects demand. Working days show stronger commuting peaks, whereas weekends tend to have a more gradual pattern across the day.

3. **Box plot:** Rental-count distributions differ across seasons. The seasonal medians and spreads show that weather conditions across the year are associated with different levels of bicycle use.

4. **Scatter plot:** Rental counts generally increase as temperature becomes more comfortable, although the wide spread of points shows that temperature is not the only factor affecting demand.

5. **Histogram:** Most hourly observations are concentrated at relatively lower rental counts, with fewer observations at very high counts. This creates a right-skewed distribution.

### Conclusion

Data visualization helped uncover patterns by summarizing many observations in a form that can be compared visually. For example, a raw table may show one row at hour 8 with 458 rentals and another row at hour 14 with 169 rentals, but the individual rows do not clearly show the repeated daily pattern. The line chart makes the overall peak and low-demand hours visible immediately, while the heatmap shows how those hourly patterns differ between working days, weekends, and holidays. Therefore, visualization reveals trends, group differences, concentration, and unusual values that are difficult to recognize by inspecting raw data alone.